# The Join That Lost the Orders

**Topic:** pandas joins and groupby &nbsp;·&nbsp; **Difficulty:** Medium

An analyst reports that revenue "dropped overnight". The database did not
change, but their query did. They switched an outer join to an inner one, and every
order whose product is missing from the catalogue silently vanished.

Rebuild the report correctly. The trap is that the naive `merge` throws away
rows, and the obvious fix (filling missing prices with zero) quietly answers a
different question.

## The data (created in the next cell)

- `orders`: 5,000 rows, with columns `order_id`, `product_id`, `quantity`, `region`.
- `catalog`: prices for **most** but not all `product_id`s.
- Some products in `orders` are missing from `catalog` entirely.

## Your task

With **pandas**:

1. Attach a `price` to every order, keeping **all** orders (a left join).
2. Orders whose product has no catalogue price must be counted but must **not**
   contribute revenue: treat their revenue as missing, not as zero.
3. Set `answers` to a dict with:
   - `"n_orders_total"`: total order count (int),
   - `"n_orders_unpriced"`: orders with no catalogue price (int),
   - `"revenue_by_region"`: a length-4 array of total revenue for regions
     `["east", "north", "south", "west"]` in that order, summing only priced
     orders.

## Evaluation

Counts must match exactly, and revenue is compared to a relative tolerance of 1e-6.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(7)
n = 5000
product_ids = np.arange(1, 301)
orders = pd.DataFrame({
    "order_id": np.arange(n),
    "product_id": rng.choice(product_ids, n),
    "quantity": rng.integers(1, 12, n),
    "region": rng.choice(["north", "south", "east", "west"], n),
})
priced = rng.choice(product_ids, 260, replace=False)   # 40 products unpriced
catalog = pd.DataFrame({
    "product_id": np.sort(priced),
    "price": np.round(rng.uniform(3.0, 90.0, len(priced)), 2),
})
print(f"orders {orders.shape}, catalog {catalog.shape}")
print(f"products in orders missing from catalog: "
      f"{orders['product_id'].nunique() - catalog['product_id'].nunique()}")

In [ ]:
# =====================================================================
# YOUR SOLUTION GOES HERE
# =====================================================================
# Use the training data created above to build your model, then create
# the variable(s) the problem statement asks for (see "Your task").
# When you are done, run the evaluation cell below to check your work.

In [ ]:
# =====================================================================
# EVALUATION: run this cell to score your answer
# =====================================================================
import numpy as np

if "answers" not in dir():
    raise NameError("Define `answers` in the solution cell before running this cell.")
if not isinstance(answers, dict):
    raise TypeError("`answers` must be a dict.")

def _digest(a):
    a = np.asarray(a, dtype=float).ravel()
    if a.size == 0:
        return 0.0
    probe = np.random.default_rng(9176).standard_normal(a.size)
    return float(a @ probe)

_CHECKS = [
    ("n_orders_total", [], 5000.0, -837.1900740724958, 1e-06),
    ("n_orders_unpriced", [], 655.0, -109.67189970349695, 1e-06),
    ("revenue_by_region", [4], 1224383.81, 54765.79658418667, 1e-06),
]

print("=" * 56)
_ok = True
for _key, _shape, _sum, _probe, _tol in _CHECKS:
    if _key not in answers:
        print(f"  {_key:<24} MISSING")
        _ok = False
        continue
    _a = np.asarray(answers[_key], dtype=float)
    if list(_a.shape) != _shape:
        print(f"  {_key:<24} shape {list(_a.shape)}, expected {_shape}")
        _ok = False
        continue
    if not np.all(np.isfinite(_a)):
        print(f"  {_key:<24} contains NaN or inf")
        _ok = False
        continue
    _g = (abs(_a.sum() - _sum) <= _tol * (1.0 + abs(_sum))
          and abs(_digest(_a) - _probe) <= _tol * (1.0 + abs(_probe)))
    print(f"  {_key:<24} {'ok' if _g else 'wrong'}")
    _ok = _ok and _g

print("=" * 56)
if _ok:
    print("PASSED: nice work!")
else:
    print("NOT PASSED yet: revisit your solution and try again.")

## Solution

`how="left"` keeps every order and leaves `price` as `NaN` where the catalogue
has no row. `quantity * price` propagates the `NaN`, and pandas' `groupby.sum`
skips `NaN` by default, so unpriced orders are counted in step 2 but excluded
from revenue without ever being turned into zeros.

```python
import numpy as np

merged = orders.merge(catalog, on="product_id", how="left")
merged["revenue"] = merged["quantity"] * merged["price"]

by_region = merged.groupby("region")["revenue"].sum()
order = ["east", "north", "south", "west"]

answers = {
    "n_orders_total": int(len(merged)),
    "n_orders_unpriced": int(merged["price"].isna().sum()),
    "revenue_by_region": by_region.reindex(order).to_numpy(),
}
```